In [ ]:
import uuid
import ipywidgets as widgets
import pandas as pd
import qnexus as qnx
from IPython.display import display

from funciones import guardar_ejecucion_csv

PROJECT_NAME = "guppy-kernel-encoding"          # Nombre exacto del proyecto existente
ALLOW_NEW_EXECUTION = False                     # False: consulta segura, no envia jobs
EXECUTION_TARGET = "local"                    # Solo se usa si ALLOW_NEW_EXECUTION = True
n_shots = 100

RESULT_SOURCE = None
local_result = None
local_counts = None
local_run_id = None
sim_job_ref = None
sim_result = None
sim_counts = None
sim_result_ids = None
LOADED_FINISHED_JOB = False
SKIP_ALL_EXECUTION = not ALLOW_NEW_EXECUTION
suffix = uuid.uuid4().hex[:8]

qnx.login()
project = qnx.projects.get(name=PROJECT_NAME)    # Consulta; nunca crea otro proyecto
qnx.context.set_active_project(project)

print("Conexion con Nexus comprobada.")
print(f"Proyecto: {PROJECT_NAME}")
print(f"Project ID: {project.id}")
print("Envio de nuevas ejecuciones:", "habilitado" if ALLOW_NEW_EXECUTION else "bloqueado")


In [2]:
# Consulta de solo lectura: recupera todos los jobs de ejecucion del proyecto.
execution_job_refs = list(qnx.jobs.get_all(
    project=project,
    job_type=[qnx.jobs.JobType.EXECUTE],
    created_after=None,
    page_size=100
))

print(f"Jobs disponibles en {PROJECT_NAME}: {len(execution_job_refs)}")
selector_options = []

for index, job in enumerate(execution_job_refs):
    job_name = getattr(job.annotations, "name", None) or str(job.id)
    job_status = getattr(job.last_status, "value", str(job.last_status))
    print(f"[{index}] {job_name} | {job_status} | id={job.id}")
    selector_options.append((f"[{index}] {job_name} | {job_status}", index))

job_selector = widgets.Dropdown(
    options=selector_options,
    value=selector_options[0][1] if selector_options else None,
    description="Job:",
    disabled=not selector_options,
    layout=widgets.Layout(width="700px")
)

display(job_selector)
print("Selecciona un job y luego ejecuta la siguiente celda.")


Jobs disponibles en guppy-kernel-encoding: 2
[0] encoding-selene-sim-f90a0e52 | COMPLETED | id=cb78ad0e-5c4e-483f-a0b3-35d4d5d2c744
[1] encoding-selene-sim-1b6a604a | COMPLETED | id=3b8e60d6-4248-4f0f-9021-9c22f50727f1


Dropdown(description='Job:', layout=Layout(width='700px'), options=(('[0] encoding-selene-sim-f90a0e52 | COMPL…

Selecciona un job y luego ejecuta la siguiente celda.


In [3]:
# Lee el indice elegido en la lista desplegable de la celda anterior.
SELECTED_JOB_INDEX = job_selector.value

if SELECTED_JOB_INDEX is None:
    raise ValueError("No hay jobs disponibles para seleccionar.")
if not 0 <= SELECTED_JOB_INDEX < len(execution_job_refs):
    raise IndexError(f"Indice fuera de rango: {SELECTED_JOB_INDEX}")

selected_job_ref = execution_job_refs[SELECTED_JOB_INDEX]
selected_job_name = getattr(selected_job_ref.annotations, "name", None) or str(selected_job_ref.id)
selected_job_status = getattr(selected_job_ref.last_status, "value", str(selected_job_ref.last_status))

print(f"Indice seleccionado: {SELECTED_JOB_INDEX}")
print(f"Job: {selected_job_name}")
print(f"Job ID: {selected_job_ref.id}")
print(f"Estado: {selected_job_status}")

if selected_job_ref.last_status != qnx.jobs.JobStatusEnum.COMPLETED:
    raise RuntimeError("El job seleccionado aun no esta COMPLETED y no tiene resultados finales.")

selected_result_refs = list(qnx.jobs.results(selected_job_ref))
if not selected_result_refs:
    raise RuntimeError(f"El job {selected_job_ref.id} no contiene resultados descargables.")

selected_downloaded_results = [ref.download_result() for ref in selected_result_refs]
selected_counts = [result.collated_counts() for result in selected_downloaded_results]

result_rows = []
for result_index, (result_ref, counts) in enumerate(zip(selected_result_refs, selected_counts)):
    total_shots = sum(counts.values())
    for outcome, count in counts.items():
        outcome_text = " | ".join(
            f"{register}={value}" for register, value in outcome
        ) if isinstance(outcome, tuple) else str(outcome)
        result_rows.append({
            "result_index": result_index,
            "result_id": str(result_ref.id),
            "outcome": outcome_text,
            "count": count,
            "shots": total_shots,
            "proportion": count / total_shots if total_shots else 0.0,
        })

selected_results_df = pd.DataFrame(result_rows)
display(selected_results_df)

sim_result = selected_downloaded_results[0]
sim_counts = selected_counts[0] if len(selected_counts) == 1 else selected_counts
RESULT_SOURCE = "selected_nexus_job"
LOADED_FINISHED_JOB = True
SKIP_ALL_EXECUTION = True                       # Elegir un job nunca dispara otro nuevo


Indice seleccionado: 0
Job: encoding-selene-sim-f90a0e52
Job ID: cb78ad0e-5c4e-483f-a0b3-35d4d5d2c744
Estado: COMPLETED


,result_index,result_id,outcome,count,shots,proportion
0,0,fc796533-6755-49e4-95ff-7b177b3b9e8e,logical[0]=1 | logical[1]=1,44,100,0.44
1,0,fc796533-6755-49e4-95ff-7b177b3b9e8e,logical[0]=0 | logical[1]=0,56,100,0.56


In [4]:
from guppylang import guppy
from guppylang.std.builtins import result
from guppylang.std.quantum import cx, h, measure, qubit


@guppy
def encode_logical_plus() -> None:
    data, parity = qubit(), qubit()
    h(data)
    cx(data, parity)
    result("logical[0]", measure(data))
    result("logical[1]", measure(parity))


encode_logical_plus.check()


In [5]:
# Compila y sube HUGR solo si realmente se va a enviar un job nuevo a Nexus/Selene.
# En modo local, en modo SKIP o al reutilizar un job terminado, esta celda no sube nada.

hugr_binary = None
ref_hugr = None

if SKIP_ALL_EXECUTION:                                           # Ya se cargo resultado o la plantilla esta en SKIP
    print("No se compila/sube HUGR porque no se enviara un job nuevo.")

elif EXECUTION_TARGET == "nexus_selene":                         # Solo Nexus necesita HUGR remoto
    hugr_binary = encode_logical_plus.compile()                  # Compila Guppy a HUGR

    ref_hugr = qnx.hugr.upload(                                  # Sube el HUGR a Nexus
        hugr_package=hugr_binary,
        name=f"encoding-logical-plus-{suffix}"
    )

    print("HUGR compilado y subido a Nexus:", ref_hugr)

elif EXECUTION_TARGET == "local":                                # La simulacion local no requiere upload
    print("Modo local: no se sube HUGR a Nexus.")

else:
    raise ValueError('EXECUTION_TARGET debe ser "local" o "nexus_selene"')


No se compila/sube HUGR porque no se enviara un job nuevo.


In [ ]:
# Ejecuta segun la decision idempotente:
#   - SKIP o job terminado cargado: no hace nada
#   - local: corre simulador local
#   - nexus_selene: envia job remoto y termina inmediatamente

if SKIP_ALL_EXECUTION:
    if LOADED_FINISHED_JOB:
        print("Se reutilizo un job terminado; se omite simulacion nueva.")
    else:
        print("Modo seguro activo; no se ejecuta simulacion local ni remota.")

elif EXECUTION_TARGET == "local":                                # Camino rapido: sin subir a queue
    local_result = (                                              # Guarda el resultado local
        encode_logical_plus                                       # Usa la funcion Guppy definida arriba
        .emulator(n_qubits=2)                                     # Crea un emulador local con 2 qubits
        .stabilizer_sim()                                         # Usa simulador estabilizador para H y CNOT
        .with_shots(n_shots)                                      # Ejecuta n_shots repeticiones
        .with_seed(42)                                            # Fija semilla para reproducibilidad
        .run()                                                    # Corre la simulacion local
    )

    RESULT_SOURCE = "local"
    local_run_id = f"local-{uuid.uuid4().hex[:12]}"           # Id unico de esta ejecucion local
    local_counts = local_result.register_counts()                 # Extrae conteos para usarlos en el notebook
    print("Simulacion local finalizada correctamente.")
    local_counts                                                  # Muestra los conteos agrupados

elif EXECUTION_TARGET == "nexus_selene":                         # Camino remoto: Selene en Nexus
    if ref_hugr is None:
        raise RuntimeError("ref_hugr no existe; ejecuta primero la celda de compilacion/upload.")

    sim_config = qnx.models.SeleneConfig(                         # Configura el simulador Selene en Nexus
        n_qubits=2,                                                # Tu encoding usa 2 qubits fisicos
        simulator=qnx.models.StabilizerSimulator()                 # Rapido para H, CNOT y circuitos Clifford
    )

    sim_job_ref = qnx.start_execute_job(                           # Envia el HUGR al simulador remoto
        programs=[ref_hugr],                                       # Usa el programa Guppy compilado y subido
        n_shots=[n_shots],                                         # Ejecuta n_shots repeticiones
        backend_config=sim_config,                                 # Usa la configuracion del simulador
        name=f"encoding-selene-sim-{suffix}"                       # Nombre unico del job
    )

    submit_status = qnx.jobs.status(sim_job_ref)                   # Consulta estado inmediatamente despues del submit

    print("Job enviado a Selene/Nexus correctamente.")
    print("sim_job_ref:", sim_job_ref)
    print("Estado inicial:", submit_status.status)
    print("Mensaje:", submit_status.message)
    print("La celda termina aqui; consulta el avance en la siguiente celda.")

else:
    raise ValueError('EXECUTION_TARGET debe ser "local" o "nexus_selene"')


In [ ]:
# Consulta el estado de un job remoto nuevo sin bloquear el notebook.
# Reejecuta esta celda manualmente cada vez que quieras revisar avance.

if sim_job_ref is None:                                           # No hay job nuevo enviado en esta sesion
    print("No hay sim_job_ref que consultar. Puede que estes en local, SKIP o usando un job ya terminado.")

else:
    status = qnx.jobs.status(sim_job_ref)                         # Consulta estado actual del job en Nexus

    print("Estado:", status.status)                               # Imprime estado resumido
    print("Mensaje:", status.message)                             # Imprime mensaje descriptivo

    queue_position = getattr(status, "queue_position", None)       # Lee posicion en cola si existe
    if queue_position is not None:
        print("Posicion en cola:", queue_position)

    if "COMPLETED" in str(status.status):                          # Si termino, descarga resultados
        sim_result_refs = list(qnx.jobs.results(sim_job_ref))      # Todas las referencias de resultado
        sim_downloaded_results = [ref.download_result() for ref in sim_result_refs]
        sim_counts_list = [r.collated_counts() for r in sim_downloaded_results]
        sim_result_ids = [str(ref.id) for ref in sim_result_refs]  # Ids para completar el registro

        sim_result = sim_downloaded_results[0]                     # Compatibilidad: primer resultado
        sim_counts = sim_counts_list[0] if len(sim_counts_list) == 1 else sim_counts_list
        RESULT_SOURCE = "new_nexus_job"
        print("Job finalizado. Resultado descargado en sim_result; conteos en sim_counts.")
        sim_counts

    elif "ERROR" in str(status.status) or "CANCELLED" in str(status.status):
        raise RuntimeError(f"El job termino sin exito: {status}")

    else:
        print("Job aun no finalizado. Vuelve a ejecutar esta celda mas tarde.")


In [ ]:
# Utilidad final: guarda los resultados de la ejecucion actual en un CSV con formato unico.
# Cubre los tres casos: simulador local, job de Nexus reutilizado (seleccionado hoy) y
# job de Nexus recien terminado con el cuaderno abierto. Todos comparten el mismo formato
# de tabla (el de Nexus) para poder analizarlos juntos. Los CSV se guardan en data/runs/.
# Para jobs de Nexus el archivo se nombra por el id del job, asi que volver a guardar el
# mismo job REEMPLAZA su CSV en vez de duplicarlo.

if RESULT_SOURCE == "local":
    if local_counts is None:
        raise RuntimeError("No hay resultados locales que guardar (local_counts esta vacio).")
    ruta_run = guardar_ejecucion_csv(
        counts=local_counts,
        source="local",
        n_shots=n_shots,
        run_id=local_run_id,
    )

elif RESULT_SOURCE == "selected_nexus_job":
    ruta_run = guardar_ejecucion_csv(
        counts=selected_counts,
        source="selected_nexus_job",
        job_id=selected_job_ref.id,
        job_name=selected_job_name,
        result_ids=[str(ref.id) for ref in selected_result_refs],
        n_shots=n_shots,
    )

elif RESULT_SOURCE == "new_nexus_job":
    ruta_run = guardar_ejecucion_csv(
        counts=sim_counts,
        source="new_nexus_job",
        job_id=sim_job_ref.id,
        job_name=getattr(sim_job_ref.annotations, "name", None) or str(sim_job_ref.id),
        result_ids=sim_result_ids,
        n_shots=n_shots,
    )

else:
    raise RuntimeError(
        "No hay una ejecucion cargada para guardar. Corre una simulacion local, "
        "selecciona un job de Nexus, o espera a que termine un job nuevo."
    )

print(f"Ejecucion guardada en: {ruta_run}")
pd.read_csv(ruta_run)
